<a href="https://colab.research.google.com/github/nnknishant/Pyspark_Project/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install the Libraries

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("demo").getOrCreate()
from pyspark.sql.functions import col, when, sum, first
from pyspark.sql import Window



In [ ]:
# Create DataFrame

Jobs_skills = [
    (1, 'Data Engineer', 'python'), (2, None, 'SQL'), (3, 'Web Developer', 'python')]

Job_Schema = "role_id int, job_role string, skills string"

Jobs_df = spark.createDataFrame(data = Jobs_skills, schema = Job_Schema)


In [ ]:
# To see the complete datasets.
Jobs_df.show()


+-------+-------------+------+
|role_id|     job_role|skills|
+-------+-------------+------+
|      1|Data Engineer|python|
|      2|         NULL|   SQL|
|      3|Web Developer|python|
+-------+-------------+------+



In [13]:

# Create the flag column

df1 = Jobs_df.withColumn("flag", when(col("job_role").isNotNull(),1).otherwise(0))
df1.show()




+-------+-------------+------+----+
|role_id|     job_role|skills|flag|
+-------+-------------+------+----+
|      1|Data Engineer|python|   1|
|      2|         NULL|   SQL|   0|
|      3|Web Developer|python|   1|
+-------+-------------+------+----+



In [15]:
# Create group of flag



df2 = df1.withColumn("group", sum(col("flag")).over(Window.partitionBy("role_id")))
df2.show()


+-------+-------------+------+----+-----+
|role_id|     job_role|skills|flag|group|
+-------+-------------+------+----+-----+
|      1|Data Engineer|python|   1|    1|
|      2|         NULL|   SQL|   0|    0|
|      3|Web Developer|python|   1|    1|
+-------+-------------+------+----+-----+



In [22]:
df_answer= df2.withColumn("Job_Role", first(col("job_role")).over(Window.partitionBy(col("group")).orderBy(col("role_id"))))
df_answer.show()




+-------+-------------+------+----+-----+
|role_id|     Job_Role|skills|flag|group|
+-------+-------------+------+----+-----+
|      2|         NULL|   SQL|   0|    0|
|      1|Data Engineer|python|   1|    1|
|      3|Data Engineer|python|   1|    1|
+-------+-------------+------+----+-----+

